### **Geo-Energy Engineering Application Final Project (Q3, 2026)**

### Project I [50% total weight]:

**Description:**

A single-phase 1D hydrogen storage reservoir (of length L) is at an initial pressure of P(x, t=0) = 100 bars everywhere, having hydrogen stored during summer. The left and right boundaries are sealed, i.e., they both exhibit "no-flow" condition. Suddenly, a power shortage happens, and we need to produce hydrogen from this reservoir from the production well located at x = L/2. The well radius is 0.15 m, with skin S = 2, and the operational pressure is set to the content value of P_BHP = 20 bars.

**Wanted:**

Simulate the pressure distribution, mass velocity field (ρ, U), and cumulative hydrogen gas production from the start of production until the average pressure in the reservoir reaches half of the original pressure, i.e., until P_avg = 50 bars. Simulate it for the base case grid size (given below) and also study the impact of grid size on the results by using 10 times less and 10 times more grid cells.

**Given:**

- Assume the reservoir is homogeneous with K = 1 Darcy and L = 100 m.  
- Use Δx = 1 m as the base case (leading to 100 grid cells as the base case).  
- Hydrogen density can be approximated by the ideal gas law, ρ / ρ₀ = P / P₀, where ρ₀ = 0.1 kg/m³ at p₀ = 10 bars.  
- Porosity is 0.2.  
- Assume all other necessary data.

**Report & Grading:**

1. Describe the governing equations, discretization and simulation approach **[10%]**

2. Simulate the pressure distribution: plot the results and explain and justify what they mean **[25%]**

3. Simulate the mass velocity field (ρ, U): plot the results and explain and justify what they mean **[25%]**

4. Simulate the cumulative hydrogen gas production (mass and volume): plot the results and explain and justify what they mean **[25%]**

5. Use 10 times coarser and 10 times finer grid cells and plot and discuss the impact of grid size on the results **[15%]**

---

# Solution to Project I: Hydrogen Storage Simulation

## 1. Governing Equations, Discretization and Simulation Approach

### 1.1 Governing Equations

#### Conservation of Mass

For a **single-phase compressible gas** flowing through 1D porous media, the continuity equation is:

$$\frac{\partial}{\partial t}\left(\phi \rho\right) + \frac{\partial}{\partial x}\left(\rho u\right) = q$$

where:
- $\phi$ = porosity [-]
- $\rho(p)$ = gas density [kg/m³], pressure-dependent
- $u$ = Darcy velocity [m/s]
- $q$ = mass source/sink per unit volume [kg/(m³·s)]

#### Darcy's Law

The momentum equation for slow viscous flow in porous media:

$$u = -\frac{k}{\mu}\frac{\partial p}{\partial x}$$

where:
- $k$ = absolute permeability [m²]
- $\mu$ = dynamic viscosity [Pa·s]
- $p$ = pressure [Pa]

#### Equation of State (Ideal Gas Law)

For hydrogen at reservoir conditions:

$$\rho(p) = \rho_0 \frac{p}{p_0}$$

where reference state: $\rho_0 = 0.1$ kg/m³ at $p_0 = 10$ bar.

#### Diffusion Equation for Pressure

Substituting Darcy's law and the ideal gas EOS into the continuity equation yields:

$$\phi c_g \frac{\partial p}{\partial t} = \frac{k}{\mu}\frac{\partial^2 p}{\partial x^2} + \frac{q}{\rho}$$

For an ideal gas, the compressibility is:

$$c_g = \frac{1}{\rho}\frac{\partial \rho}{\partial p} = \frac{1}{p}$$

This is the **fundamental governing equation** for this problem: a nonlinear parabolic PDE in pressure.

---

### 1.2 Discretization Scheme

#### Spatial Discretization (Finite Volume Method)

The reservoir domain $[0, L]$ is divided into $n_x$ uniform control volumes with grid spacing:

$$\Delta x = \frac{L}{n_x}$$

For the **base case**: $\Delta x = 1$ m → $n_x = 100$ cells.

**Cell-centered scheme**: pressure unknowns $p_i$ are defined at cell centers $x_i = (i - 0.5)\Delta x$ for $i = 1, \ldots, n_x$.

#### Harmonic-Average Transmissibility

The **inter-block transmissibility** between cells $i$ and $i+1$ is:

$$T_{i+\frac{1}{2}} = \frac{A}{\Delta x} \cdot \frac{2k_i k_{i+1}}{k_i + k_{i+1}}$$

where $A = \Delta y \cdot \Delta z$ is the cross-sectional area. For homogeneous permeability ($k_i = k$ for all $i$), this simplifies to:

$$T = \frac{k A}{\Delta x}$$

#### Accumulation Term

The **accumulation coefficient** for cell $i$:

$$\Theta_i = \frac{V_i \phi}{\Delta t}$$

where $V_i = \Delta x \cdot A$ is the cell bulk volume.

#### Temporal Discretization (Fully Implicit)

The discretized mass balance for cell $i$ at time level $n+1$ is:

$$\Theta_i \left(\rho_i^{n+1} - \rho_i^n\right) = F_{i-\frac{1}{2}}^{n+1} - F_{i+\frac{1}{2}}^{n+1} + Q_i^{n+1}$$

where the mass flux across face $i+\frac{1}{2}$ is:

$$F_{i+\frac{1}{2}}^{n+1} = T_{i+\frac{1}{2}} \cdot \rho_{i+\frac{1}{2}}^{n+1} \cdot \frac{p_{i+1}^{n+1} - p_i^{n+1}}{\mu}$$

**Density upwinding**: $\rho_{i+\frac{1}{2}}^{n+1}$ is taken from the upstream cell based on the pressure gradient.

Substituting $\rho = \rho_0 p / p_0$ yields a **linear system in pressure**:

$$\mathbf{M} \mathbf{p}^{n+1} = \mathbf{f}$$

where:
- $\mathbf{M}$ is a **tridiagonal matrix** (efficient to solve)
- $\mathbf{f}$ contains terms from the previous time step and well contributions

---

### 1.3 Well Model (Peaceman Formulation)

The production well at cell $i_w = n_x / 2$ (domain center) is modeled using the **Peaceman well index**:

$$Q_w = \Gamma_w \left(p_{i_w}^{n+1} - p_{bhp}\right)$$

where the **productivity index** is:

$$\Gamma_w = \frac{2\pi k h}{\mu \left[\ln(r_0 / r_w) + S\right]}$$

with:
- $r_0 = 0.208 \Delta x$ = equivalent wellbore radius for a square grid block
- $r_w = 0.15$ m = actual wellbore radius
- $h = \Delta z$ = perforation thickness (well height)
- $S = 2$ = skin factor (positive → damaged formation near wellbore)
- $p_{bhp} = 20$ bar = bottom-hole pressure constraint

The well term $Q_w$ (in units of mass/time) is added to the mass balance of cell $i_w$ as a **sink**.

---

### 1.4 Boundary Conditions

Both reservoir boundaries are **sealed (no-flow)**:

$$\left.\frac{\partial p}{\partial x}\right|_{x=0} = 0, \qquad \left.\frac{\partial p}{\partial x}\right|_{x=L} = 0$$

Numerically implemented by:
- **Left boundary** (cell 1): No flux term $F_{1/2}$ → omit connection to left
- **Right boundary** (cell $n_x$): No flux term $F_{n_x + 1/2}$ → omit connection to right

This results in a **closed system**: total mass is conserved except for well production.

---

### 1.5 Simulation Algorithm

**Step 1: Initialization**
- Define grid: $n_x$ cells, $\Delta x = L / n_x$
- Compute transmissibilities: $T = k A / \Delta x$
- Compute well productivity index: $\Gamma_w$
- Set initial conditions: $p_i^0 = 100$ bar for all cells

**Step 2: Time-Stepping Loop** 
- While $\bar{p} = \frac{1}{n_x}\sum_{i=1}^{n_x} p_i > 50$ bar:
  - **Assemble linear system**:
    - Coefficient matrix $\mathbf{M}$ (tridiagonal)
    - RHS vector $\mathbf{f} = \Theta \rho^n - \text{well term}$
  - **Solve for pressure**: $\mathbf{p}^{n+1} = \mathbf{M}^{-1} \mathbf{f}$
  - **Update density**: $\rho_i^{n+1} = \rho_0 p_i^{n+1} / p_0$
  - **Compute mass velocity**: $(\rho u)_{i+1/2} = T \rho_{i+1/2} (p_{i+1} - p_i) / \mu$
  - **Track cumulative production**:
    - Mass: $M_{cum} += Q_w \rho_{i_w} \Delta t$
    - Volume (STP): $V_{cum} += Q_w (p_{i_w} / p_0) \Delta t$
  - **Increment time**: $t \leftarrow t + \Delta t$

**Step 3: Post-Processing**
- Plot pressure profiles $p(x, t)$ at selected times
- Plot mass velocity $\rho u(x, t)$
- Plot cumulative production $M_{cum}(t)$ and $V_{cum}(t)$

**Adaptive Time-Stepping**: Start with small $\Delta t$ (e.g., 0.01 day), increase by factor 1.2 if convergence is good, cap at $\Delta t_{max} = 10$ days.

---

### 1.6 Grid Sensitivity Study

To assess **numerical convergence**, simulate with three grid resolutions:

| Case | Grid cells $n_x$ | Grid spacing $\Delta x$ |
|------|------------------|------------------------|
| Coarse | 10 | 10 m |
| **Base** | **100** | **1 m** |
| Fine | 1000 | 0.1 m |

Compare:
- Pressure depletion curves
- Production rates
- Mass balance errors

---

### Notation Summary

| Symbol | Description | Unit |
|--------|-------------|------|
| $p$ | Pressure | [bar] |
| $\rho$ | Gas density | [kg/m³] |
| $\phi$ | Porosity | 0.2 [-] |
| $k$ | Permeability | 1 Darcy = $9.87 \times 10^{-13}$ m² |
| $\mu$ | Hydrogen viscosity | $\approx 8.9 \times 10^{-6}$ Pa·s |
| $c_g$ | Gas compressibility | $1/p$ [bar⁻¹] |
| $T$ | Transmissibility | [m³/day/bar] |
| $\Gamma_w$ | Well productivity index | [m³/day/bar] |
| $\Delta x$ | Grid spacing | [m] |
| $\Delta t$ | Time step | [day] |
| $Q_w$ | Well production rate | [m³/day] at reservoir conditions |
| $r_w$ | Wellbore radius | 0.15 m |
| $S$ | Skin factor | 2 (damaged) |
| $L$ | Reservoir length | 100 m |
| $p_0$ | Reference pressure | 10 bar |
| $\rho_0$ | Reference density | 0.1 kg/m³ |

---